In [1]:
########## import some useful package ##########

import time
t1 = time.time()

import uproot
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import h5py

import ripser
import persim

In [2]:
########## read the data ##########

processes = ["jjBG_ptcut", "ttBG_ptcut", "bbBG_ptcut", "wwBG", "wwvvBG", "zzBG", "zzvvBG", "hzvvBG"]
event_type = [str(i)+"_500K" for i in processes]

BR_hbb = 0.5824
Xsection = [1.08143e-1, 3.539e-2, 1.86879e-2, 2.200885e+1, 7.053723e+1, 1.3355, 3.983523e+1, 7.03508147*BR_hbb]     ### unit: fb
Luminosity = 1000   ### unit: fb^-1
simulation_num = 500000

features = ["VLCjetR10N2", "VLCjetR10N2.PT", "VLCjetR10N2.Eta", "VLCjetR10N2.Phi", "VLCjetR10N2.Mass", "MissingET.MET", "VLCjetR10N2.BTag", "EFlow.Eta", "EFlow.Phi", "EFlow.ET"]

##### set data path #####

event_path = []
for type in event_type:
    event_path.append("/data/mucollider/two_boosted/10TeV/" + type + "/delphes_output.root")

##### get the data file #####

data_file =[]
for path in event_path:
    data_file.append(uproot.open(path))
    
##### read data with features #####
events = []     ### total events
for process in processes:
    tmp_events = []
    for feature in features:
        tmp_events.append(data_file[processes.index(process)]["Delphes;1"][feature].array())
    tmp_events = np.expand_dims(tmp_events, axis=-1)
    tmp_events = tmp_events.transpose((1,0,2))
    tmp_events = np.squeeze(tmp_events,axis=(2,))
    events.append(tmp_events)
    print("Time:{:^8.4f}(s)".format(time.time()-t1))
del tmp_events

Time:10.8745 (s)
Time:21.3966 (s)
Time:31.1145 (s)
Time:36.3801 (s)
Time:42.3518 (s)
Time:47.8679 (s)
Time:53.7585 (s)
Time:60.7042 (s)


In [3]:
########## define useful function ##########

##### calculate significance #####

def significance(s,b):
    return np.sqrt(2*((s+b)*np.log(1+s/b)-s))

##### count event number #####

def count(events):
    events_num = []
    for i, process in enumerate(processes):
        events_num.append(len(events[processes.index(process)]))
        print("There are", events_num[i], process, "events. Corresponding cross section:", Xsection[processes.index(process)]*events_num[i]/simulation_num, "(fb)")
        
##### select if Fat Jet >= 2 #####
        
def Fat_Jet_selection(events):
    where1 = np.where(events[:,features.index("VLCjetR10N2")]>=2)
    return events[where1]

##### select if M_jet_leading > XX GeV #####

def mass_leading_selection(events, low_mass_cut, high_mass_cut):
    where1 = []
    for i in range(len(events)):
        switch=1
        if events[i][features.index("VLCjetR10N2.Mass")][0]<low_mass_cut:
            switch=0
        if events[i][features.index("VLCjetR10N2.Mass")][0]>high_mass_cut:
            switch=0
        if switch==1:
            where1.append(i)
    return events[where1]

##### select if M_jet_subleading > XX GeV #####

def mass_subleading_selection(events, low_mass_cut, high_mass_cut):
    where1 = []
    for i in range(len(events)):
        switch=1
        if events[i][features.index("VLCjetR10N2.Mass")][1]<low_mass_cut:
            switch=0
        if events[i][features.index("VLCjetR10N2.Mass")][1]>high_mass_cut:
            switch=0
        if switch==1:
            where1.append(i)
    return events[where1]

##### select if pT_J_leading > XX GeV #####

def pT_leading_selection(events, low_pT_cut, high_pT_cut):
    where1 = []
    for i in range(len(events)):
        switch = 1
        if events[i][features.index("VLCjetR10N2.PT")][0]<low_pT_cut:
            switch=0
        if events[i][features.index("VLCjetR10N2.PT")][0]>high_pT_cut:
            switch=0
        if switch==1:
            where1.append(i)
    return events[where1]

##### select if pT_J_subleading > XX GeV #####

def pT_subleading_selection(events, low_pT_cut, high_pT_cut):
    where1 = []
    for i in range(len(events)):
        switch = 1
        if events[i][features.index("VLCjetR10N2.PT")][1]<low_pT_cut:
            switch=0
        if events[i][features.index("VLCjetR10N2.PT")][1]>high_pT_cut:
            switch=0
        if switch==1:
            where1.append(i)
    return events[where1]

##### define X_HH (defined in 2207.09602v2) #####

def calculate_X_HH(event):
    jet_mass1 = event[features.index("VLCjetR10N2.Mass")][0]
    jet_mass2 = event[features.index("VLCjetR10N2.Mass")][1]
    diff1 = np.abs(jet_mass1 - 124)
    diff2 = np.abs(jet_mass2 - 124)
    if diff1<diff2:
        m1 = jet_mass1
        m2 = jet_mass2
    else:
        m1 = jet_mass2
        m2 = jet_mass1
    return np.sqrt(((m1-124)/(0.1*m1+0.00001))**2 + ((m2-115)/(0.1*m2+0.00001))**2)

##### calculate M_JJ #####

def calculate_M_JJ(event):
    p = [0,0,0,0]    ### four momentum
    for i in range(2):    ### leading jet and sub-leading jet
        pt = event[features.index("VLCjetR10N2.PT")][i]
        eta = event[features.index("VLCjetR10N2.Eta")][i]
        phi = event[features.index("VLCjetR10N2.Phi")][i]
        mass = event[features.index("VLCjetR10N2.Mass")][i]
        p[1] = p[1] + pt*np.cos(phi)    ### px
        p[2] = p[2] + pt*np.sin(phi)    ### py
        p[3] = p[3] + pt*np.sinh(eta)    ### pz
        p[0] = p[0] + np.sqrt( mass**2 + (pt*np.cos(phi))**2 + (pt*np.sin(phi))**2 + (pt*np.sinh(eta))**2 )    ### energy
    M_JJ = np.sqrt(p[0]**2 - p[1]**2 - p[2]**2 - p[3]**2)
    return M_JJ

##### calculate pT_JJ #####

def calculate_pT_JJ(event):
    p = [0,0,0,0]    ### four momentum
    for i in range(2):    ### leading jet and sub-leading jet
        pt = event[features.index("VLCjetR10N2.PT")][i]
        eta = event[features.index("VLCjetR10N2.Eta")][i]
        phi = event[features.index("VLCjetR10N2.Phi")][i]
        mass = event[features.index("VLCjetR10N2.Mass")][i]
        p[1] = p[1] + pt*np.cos(phi)    ### px
        p[2] = p[2] + pt*np.sin(phi)    ### py
        p[3] = p[3] + pt*np.sinh(eta)    ### pz
        p[0] = p[0] + np.sqrt( mass**2 + (pt*np.cos(phi))**2 + (pt*np.sin(phi))**2 + (pt*np.sinh(eta))**2 )    ### energy
    pT_JJ = np.sqrt(p[1]**2 + p[2]**2)
    return pT_JJ

##### calculate \Delta_\Eta #####

def calculate_dEta_JJ(event):
    return np.abs(event[features.index("VLCjetR10N2.Eta")][0] - event[features.index("VLCjetR10N2.Eta")][1])

##### calculate TDA #####

def calculate_TDA(event):
#     neutrino_ID = [12, 14, 16, -12, -14, -16]
#     where_jet_particle = np.where(event[features.index("Particle.Status")]==1)[0]
#     where_neutrino = np.where(np.isin(event[features.index("Particle.PID")], neutrino_ID))[0]
#     where_jet_particle = np.delete(where_jet_particle, np.isin(where_jet_particle, where_neutrino))

    particle_ET = event[features.index("EFlow.ET")]
    particle_ETcut = 0
    where_ET_larger = np.where(particle_ET>particle_ETcut)
    particle_ET = event[features.index("EFlow.ET")][where_ET_larger]
    
    center_eta = (event[features.index("VLCjetR10N2.Eta")][0]+event[features.index("VLCjetR10N2.Eta")][1])/2    ### centerize the event
    center_phi = (event[features.index("VLCjetR10N2.Phi")][0]+event[features.index("VLCjetR10N2.Phi")][1])/2
    particle_eta = event[features.index("EFlow.Eta")][where_ET_larger]-center_eta
    particle_phi = event[features.index("EFlow.Phi")][where_ET_larger]-center_phi
    
    where_larger_pi = np.where(particle_phi>np.pi)
    particle_phi[where_larger_pi] = -(2*np.pi - particle_phi[where_larger_pi])    ### check if jet particle split
    where_smaller_mpi = np.where(particle_phi<-np.pi)
    particle_phi[where_smaller_mpi] = 2*np.pi + particle_phi[where_smaller_mpi]    ### check if jet particle split

    P = np.array([particle_eta, particle_phi]).T   ### calculate TDA
    diagrams = ripser.ripser(P)['dgms']
    
    return diagrams

##### calculate TDA sum of lifetime #####

def TDA_sum(diagrams):
    sum_l0 = np.sum(diagrams[0][:-1,1] - diagrams[0][:-1,0])
    sum_l1 = np.sum(diagrams[1][:,1] - diagrams[1][:,0])
    return sum_l0, sum_l1

##### calculate TDA mean of lifetime #####

def TDA_mean(diagrams):
    l0 = diagrams[0][:-1,1]-diagrams[0][:-1,0]
    l1 = diagrams[1][:,1]-diagrams[1][:,0]
    if np.sum(l1)==0:
        l1 = np.concatenate([l1,[0]])
    return np.mean(l0), np.mean(l1)

##### calculate TDA entropy #####

def TDA_entropy(diagrams):
    l0 = diagrams[0][:-1,1]-diagrams[0][:-1,0]
    L0 = np.sum(l0)
    S0 = (l0/L0)*np.log2((l0/L0))
    l1 = diagrams[1][:,1]-diagrams[1][:,0]
    L1 = np.sum(l1)
    S1 = (l1/L1)*np.log2((l1/L1))
    return -np.sum(S0), -np.sum(S1)

##### calculate TDA Lifetime*Birthtime #####

def TDA_LB(diagrams):
    return np.sum(diagrams[1][:,0]*(diagrams[1][:,1]-diagrams[1][:,0]))

In [4]:
########## preselection ##########

print("Before any selection:")
count(events)

##### 2 fat jet selection #####

for process in processes:
    events[processes.index(process)] = Fat_Jet_selection(events[processes.index(process)])
print("\nAfter 2 fat jet selection:")
count(events)

##### leading jet pT selection #####

leading_low_pT_cut = 200   ### GeV
leading_high_pT_cut = 800   ### GeV
for process in processes:
    events[processes.index(process)] = pT_leading_selection(events[processes.index(process)], leading_low_pT_cut, leading_high_pT_cut)
print("\nAfter leading jet pT selection:")
count(events)

##### subleading jet pT selection #####

subleading_low_pT_cut = 200   ### GeV
subleading_high_pT_cut = 600   ### GeV
for process in processes:
    events[processes.index(process)] = pT_subleading_selection(events[processes.index(process)], subleading_low_pT_cut, subleading_high_pT_cut)
print("\nAfter subleading jet pT selection:")
count(events)

##### leading jet mass selection #####

leading_low_mass_cut = 65
leading_high_mass_cut = 150
for process in processes:
    events[processes.index(process)] = mass_leading_selection(events[processes.index(process)], leading_low_mass_cut, leading_high_mass_cut)
print("\nAfter leading jet mass selection:")
count(events)

##### subleading jet mass selection #####

subleading_low_mass_cut = 65
subleading_high_mass_cut = 150
for process in processes:
    events[processes.index(process)] = mass_subleading_selection(events[processes.index(process)], subleading_low_mass_cut, subleading_high_mass_cut)
print("\nAfter subleading jet mass selection:")
count(events)

Before any selection:
There are 500000 jjBG_ptcut events. Corresponding cross section: 0.108143 (fb)
There are 500000 ttBG_ptcut events. Corresponding cross section: 0.03539 (fb)
There are 500000 bbBG_ptcut events. Corresponding cross section: 0.0186879 (fb)
There are 500000 wwBG events. Corresponding cross section: 22.00885 (fb)
There are 500000 wwvvBG events. Corresponding cross section: 70.53723 (fb)
There are 500000 zzBG events. Corresponding cross section: 1.3355 (fb)
There are 500000 zzvvBG events. Corresponding cross section: 39.83523 (fb)
There are 500000 hzvvBG events. Corresponding cross section: 4.097231448128 (fb)

After 2 fat jet selection:
There are 499967 jjBG_ptcut events. Corresponding cross section: 0.10813586256200002 (fb)
There are 499969 ttBG_ptcut events. Corresponding cross section: 0.03538780582 (fb)
There are 499963 bbBG_ptcut events. Corresponding cross section: 0.0186865170954 (fb)
There are 301339 wwBG events. Corresponding cross section: 13.264249700299999 

In [5]:
##### calculate the high level features #####

num_of_events = []

m_J_leading = []
m_J_subleading = []
pt_J_leading = []
pt_J_subleading = []
missET = []
X_HH = []
M_JJ = []
pT_JJ = []
dEta_JJ = []
BTag_leading = []
BTag_subleading = []

for process in processes:
    num_of_events.append(len(events[processes.index(process)]))
    for i in range(len(events[processes.index(process)])):
        m_J_leading.append(events[processes.index(process)][i][features.index("VLCjetR10N2.Mass")][0])
        m_J_subleading.append(events[processes.index(process)][i][features.index("VLCjetR10N2.Mass")][1])
        pt_J_leading.append(events[processes.index(process)][i][features.index("VLCjetR10N2.PT")][0])
        pt_J_subleading.append(events[processes.index(process)][i][features.index("VLCjetR10N2.PT")][1])
        missET.append(events[processes.index(process)][i][features.index("MissingET.MET")][0])
        X_HH.append(calculate_X_HH(events[processes.index(process)][i]))
        M_JJ.append(calculate_M_JJ(events[processes.index(process)][i]))
        pT_JJ.append(calculate_pT_JJ(events[processes.index(process)][i]))
        dEta_JJ.append(calculate_dEta_JJ(events[processes.index(process)][i]))
        BTag_leading.append(events[processes.index(process)][i][features.index("VLCjetR10N2.BTag")][0])
        BTag_subleading.append(events[processes.index(process)][i][features.index("VLCjetR10N2.BTag")][1])
    print("Time:{:^8.4f}(s)".format(time.time()-t1))

Time:98.6663 (s)
Time:100.7155(s)
Time:101.8949(s)
Time:101.8958(s)
Time:108.4201(s)
Time:108.4224(s)
Time:116.7955(s)
Time:124.5542(s)


In [6]:
##### calculate TDA high level features #####

H_0 = []    ### sum of 0'th order lifetime 
H_1 = []    ### sum of 1'th order lifetime 
S_0 = []    ### sum of 0'th order entropy
S_1 = []    ### sum of 1'th order entropy
LB_1 = []   ### sum of 1'th order Lifetime*Birthtime

for process in processes:
    for i in range(len(events[processes.index(process)])):
        if (i%20000)==19999:
            print("check point")
        TDA_tmp = calculate_TDA(events[processes.index(process)][i])
        H_0.append(TDA_sum(TDA_tmp)[0])        
        H_1.append(TDA_sum(TDA_tmp)[1])
        S_0.append(TDA_entropy(TDA_tmp)[0])   
        S_1.append(TDA_entropy(TDA_tmp)[1])
        LB_1.append(TDA_LB(TDA_tmp))
    print("Time:{:^8.4f}(s)".format(time.time()-t1))

Time:233.8979(s)
Time:437.3740(s)
Time:557.9838(s)
Time:557.9946(s)
check point
check point
Time:658.1565(s)
Time:658.1778(s)
check point
check point
check point
Time:802.4052(s)
check point
check point
check point
Time:971.4653(s)


In [7]:
##### make target #####

labels = np.arange(len(num_of_events))
target_everytype = np.repeat(labels, num_of_events)    ### this target 0~8 represent ["hhvv", "hhmumu", "jjBG_ptcut", "ttBG_ptcut", "bbBG_ptcut", "wwBG", "wwvvBG", "zzBG", "zzvvBG"]

target_sigbg = np.zeros(sum(num_of_events))
# sig_num = num_of_events[0]+num_of_events[1]
# target_sigbg[0:sig_num] = 1     ### this target 1 represent signal, 0 represent background

In [8]:
##### output test data #####

d = {'m_J_leading':m_J_leading, 'm_J_subleading':m_J_subleading, 'pt_J_leading':pt_J_leading, 'pt_J_subleading':pt_J_subleading, 'missET':missET, 'X_HH':X_HH, 'M_JJ':M_JJ, 'pT_JJ':pT_JJ, 'dEta_JJ':dEta_JJ, 'BTag_leading':BTag_leading, 'BTag_subleading':BTag_subleading, 'H_0':H_0, 'H_1':H_1, 'S_0':S_0, 'S_1':S_1, 'LB_1':LB_1, 'target_sigbg':target_sigbg, 'target_everytype':target_everytype}
df = pd.DataFrame(data=d)
df

,m_J_leading,m_J_subleading,pt_J_leading,pt_J_subleading,missET,X_HH,M_JJ,pT_JJ,dEta_JJ,BTag_leading,BTag_subleading,H_0,H_1,S_0,S_1,LB_1,target_sigbg,target_everytype
0,135.700012,138.502884,511.322784,402.989594,685.714355,1.903399,633.959788,715.366728,0.098704,0,0,23.896255,0.429113,6.143583,3.053320,0.135894,0.0,0
1,81.016571,65.882721,444.317078,378.274658,224.199966,9.150361,3395.574654,66.219950,4.196529,0,0,22.867751,0.662079,5.830665,2.059247,0.713108,0.0,0
2,132.311707,106.482826,512.989441,362.952057,323.084229,1.017057,3420.926111,565.891697,4.129205,0,0,28.428135,0.971165,6.682404,3.847509,0.602797,0.0,0
3,91.111664,66.230820,799.849182,438.360687,407.972687,8.200667,1242.037397,666.960525,1.053495,4,0,28.054530,1.265621,6.959315,2.590638,1.389085,0.0,0
4,73.055542,91.983734,333.722870,317.302002,273.336334,6.714089,2209.435677,311.199625,3.800406,0,0,22.817147,0.682578,6.305144,4.163869,0.169137,0.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
239259,125.469986,88.443695,353.622009,237.174194,109.711571,3.004904,949.020885,117.305247,2.088326,4,0,7.562420,0.086156,4.114513,2.160574,0.016097,0.0,7
239260,126.991714,77.603149,450.166595,304.349335,253.911865,4.824735,761.635937,242.256869,0.443938,7,0,9.395132,0.023997,4.810682,1.678647,0.003651,0.0,7
239261,122.611610,70.202408,625.818787,379.936096,256.022217,6.382200,3729.506546,252.299262,4.031278,6,7,9.417892,0.130752,4.169880,2.862918,0.017190,0.0,7
239262,116.044106,96.648537,774.490662,241.568985,685.338806,2.018764,1850.672045,656.360670,2.820701,7,0,11.298783,0.021792,4.127407,0.069006,0.003169,0.0,7


In [9]:
h5f = h5py.File('/data/mucollider/two_boosted/10TeV/data_HLfeature.h5', 'w')
for key, value in d.items():
    h5f.create_dataset(key, data=value)
    print(f"已寫入 Dataset: {key}")

已寫入 Dataset: m_J_leading
已寫入 Dataset: m_J_subleading
已寫入 Dataset: pt_J_leading
已寫入 Dataset: pt_J_subleading
已寫入 Dataset: missET
已寫入 Dataset: X_HH
已寫入 Dataset: M_JJ
已寫入 Dataset: pT_JJ
已寫入 Dataset: dEta_JJ
已寫入 Dataset: BTag_leading
已寫入 Dataset: BTag_subleading
已寫入 Dataset: H_0
已寫入 Dataset: H_1
已寫入 Dataset: S_0
已寫入 Dataset: S_1
已寫入 Dataset: LB_1
已寫入 Dataset: target_sigbg
已寫入 Dataset: target_everytype


In [10]:
print("Time:{:^8.4f}(s)".format(time.time()-t1))

Time:973.1068(s)


In [11]:
h5f.close()